<a href="https://colab.research.google.com/github/DhimanTarafdar/A/blob/main/Simple_RNN_notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ১. RNN কী এবং কেন এলো — Big Picture

আমরা এর আগে দুটো Neural Network পড়েছি:
- **ANN (Artificial Neural Network)** → Tabular / structured data এর জন্য
- **CNN (Convolutional Neural Network)** → Image, Video এর জন্য
কিন্তু এই দুটো দিয়ে একটা বড় ধরনের data handle করা যাচ্ছিল না — সেটা হলো **Sequential Data**।

এই সমস্যা সমাধানের জন্যই তৈরি হলো তৃতীয় ধরনের Neural Network:

> **RNN (Recurrent Neural Network)** — এটি একটি special class of Neural Network যা specifically **sequential data** এর উপর কাজ করার জন্য design করা হয়েছে।

**"Recurrent" মানে কী?**
Recurrent মানে পুনরাবৃত্তি। RNN এ একই cell বারবার apply হয় — প্রতিটা time step এ। এবং প্রতিবার সে আগের step থেকে তথ্য নিয়ে কাজ করে। এই recurrence বা পুনরাবৃত্তিই এর নামের কারণ।

---

## ২. Sequential Data কী

Sequential data মানে এমন data যেখানে **sequence (ক্রম) matter করে** — মানে কোনটা আগে এলো, কোনটা পরে এলো, সেটা গুরুত্বপূর্ণ।

**উদাহরণ ১ — Text:**
> "আমার নাম রাহুল"

এখানে প্রতিটা word একটার পর একটা আসছে। যদি এলোমেলো করে দেওয়া হয় — "রাহুল নাম আমার" — তাহলে মানে নষ্ট হয়ে যাবে।

**উদাহরণ ২ — Time Series:**
Stock market এর graph দেখো। 2020 সালে কত ছিল, 2021 তে কত হলো — এই ক্রমটা জানলেই বোঝা যায় ভবিষ্যতে কী হতে পারে। ক্রম সরিয়ে দিলে prediction অর্থহীন।

**উদাহরণ ৩ — Audio / Speech:**
তুমি যখন কথা বলছো, সেই sound এর waveform এও sequence আছে। প্রতিটা millisecond এর sound তার আগের এবং পরেরটার সাথে connected।

**উদাহরণ ৪ — Video:**
Video আসলে image এর sequence। Frame 1, Frame 2, Frame 3... — প্রতিটা frame এর মানে বোঝার জন্য আগের frame কী ছিল সেটা জানা দরকার।

**আরও উদাহরণ:** DNA sequence, আবহাওয়ার data।

> **Non-Sequential Data এর উদাহরণ:** একজন student এর CGPA, বয়স, gender দিয়ে placement হবে কি না সেটা predict করা — এখানে কোনো ক্রম নেই, তাই এটা non-sequential। এই কাজে ANN ভালো।

> **সারকথা:** Sequential data সেটাই যেখানে একটা item এর আগে বা পরে কী ছিল সেটা জানা দরকার।

---

## ৩. ANN vs RNN — পার্থক্য কোথায়

| বিষয় | ANN | RNN |
|-------|-----|-----|
| **Data type** | Tabular / fixed-size | Sequential / variable-length |
| **Input** | সব feature একসাথে | এক সময়ে একটা করে |
| **Memory** | নেই | আছে (hidden state) |
| **Time এর ধারণা** | নেই | আছে (time step) |
| **Weight** | আলাদা আলাদা | সব time step এ shared |
| **Architecture** | Feed-forward (সামনের দিকে) | Recurrent (loop আছে) |
| **উপযুক্ত কাজ** | Classification, Regression | NLP, Time series, Speech |

---

## ৪. ANN দিয়ে Sequential Data Handle করা যায় না কেন

এটা বোঝার জন্য একটু ভেবে দেখো — ANN কে যদি text দিতে চাও, তাহলে প্রথমে text কে number এ convert করতে হবে।

### Text কে Vectorize করার পদ্ধতিগুলো

ANN বা যেকোনো Neural Network শুধু number বোঝে। Text কে number এ convert করার সবচেয়ে basic পদ্ধতি: **One-Hot Encoding**

ধরো vocabulary তে ৩টা unique word আছে: [আমি, ভালো, আছি]

| Word | Vector |
|------|--------|
| আমি | [1, 0, 0] |
| ভালো | [0, 1, 0] |
| আছি | [0, 0, 1] |

যদি vocabulary তে **10,000** word থাকে, তাহলে প্রতিটা word হবে **10,000 size** এর vector।

**One-Hot Encoding এর সমস্যা:**
- Vector অনেক বড় হয়ে যায় (sparse vector)
- "রাজা" এবং "রানী" এর মধ্যে কোনো similarity নেই — দুটো সম্পূর্ণ আলাদা vector, অথচ এদের মধ্যে সম্পর্ক আছে
- কোনো semantic meaning নেই
**তাহলে কী করা হয়? → Word Embedding**

Word Embedding এ প্রতিটা word কে একটা **dense, low-dimensional** vector এ map করা হয় যেখানে similar meaning এর word গুলো vector space এ কাছাকাছি থাকে।

```
One-Hot:  "রাজা"  → [0, 0, 1, 0, 0, ..., 0]  (10,000 dimension, বেশিরভাগ 0)
Embedding: "রাজা"  → [0.2, -0.5, 0.8, 0.1]    (মাত্র 4 dimension, সব meaningful)
```

> Embedding এর বিস্তারিত আমরা NLP section এ পড়বো। এখন শুধু জেনে রাখো যে RNN এর আগে সাধারণত Embedding layer থাকে।

---

### ANN তে Sequential Data দিলে কী সমস্যা হয়

#### সমস্যা ১ — Variable Input Size

প্রতিটা sentence এ word সংখ্যা আলাদা:
- "আমার নাম রাহুল" → 3 words
- "আমি ঢাকা বিশ্ববিদ্যালয়ে পড়ি" → 4 words
ANN এর input layer এর size **fixed** থাকে। Variable size input সে নিতে পারে না।

---

#### সমস্যা ২ — Zero Padding এর সমস্যা

একটা জুগাড় solution: সবচেয়ে বড় sentence অনুযায়ী input size fix করো, ছোট গুলোতে শেষে **zero** বসিয়ে সমান করো।

কিন্তু সমস্যা:
- যদি সবচেয়ে বড় sentence এ 100 words, vocabulary 10,000 → input = **10 লাখ**
- Weights হবে কোটিরও বেশি — **অপ্রয়োজনীয় computation**
- Test এ কেউ 200 word এর sentence দিলে কী হবে?
---

#### সমস্যা ৩ — সবচেয়ে বড়: Sequence Information হারিয়ে যায়

ANN তে পুরো sentence এর সব word **একসাথে, একই মুহূর্তে** input হিসেবে ঢোকে।

তাই ANN জানতে পারে না — "ভালো" শব্দটা আগে এসেছিল নাকি পরে।

**ANN এর কোনো memory নেই।** সে বলতে পারে না "আগে X এসেছিল, তারপর Y"।

Text এর মধ্যে যে **semantic meaning** sequence থেকে আসে, ANN সেটা ধরতেই পারে না।

> **উপসংহার:** ANN দিয়ে Sequential data handle করা technically সম্ভব হলেও এটা inefficient, inaccurate এবং by design এই কাজের জন্য তৈরি না।

---

## ৫. RNN কীভাবে এই সমস্যা সমাধান করে — মূল Idea

RNN এর মূল idea টা মানুষের চিন্তার মতোই:

> তুমি যখন একটা বই পড়ো, তুমি প্রতিটা word পড়ো **একটার পর একটা**। আগের word গুলো পড়ে যে context বুঝেছো, সেটা মনে রেখে পরের word বোঝো।

RNN ঠিক এইভাবে কাজ করে:
- **এক সময়ে একটা input নেয়** → variable length এর সমস্যা নেই
- **আগের step এর তথ্য মনে রাখে** → sequence information হারায় না
- **একই weights ব্যবহার করে** → parameter বাড়ে না
এই "মনে রাখার" জিনিসটাকে বলা হয় **Hidden State**।

---

## ৬. RNN এর Architecture এবং Workflow

### মূল Components:

একটা Simple RNN cell এ থাকে:
- **Input (xₜ):** এই time step এর input (যেমন একটা word এর vector)
- **Hidden State (hₜ):** এই step এ network এর "memory" — আগের সব step এর compressed summary
- **Output (yₜ):** এই step এর output
```
        hₜ₋₁ (আগের hidden state)
          │
          ▼
xₜ ──▶ [RNN Cell] ──▶ hₜ (নতুন hidden state)
                │
                ▼
               yₜ (output)
```

### Equations:

```
hₜ = tanh(Wₓ · xₜ + Wₕ · hₜ₋₁ + b)
yₜ = Wᵧ · hₜ + bᵧ
```

**সহজ বাংলায়:**
- নতুন hidden state = (এই step এর input × Wₓ) + (আগের hidden state × Wₕ) + bias → tanh দিয়ে activate করো
- Output = নতুন hidden state × Wᵧ + output bias
### কেন tanh Activation?

RNN এ **tanh** ব্যবহারের কারণ:
- tanh এর output range **-1 থেকে +1** — এটা hidden state কে bounded রাখে
- যদি unbounded হতো, hidden state এর value ক্রমশ বড় হয়ে network অস্থির হয়ে যেত
- tanh এর negative value handle করার ক্ষমতা আছে (ReLU পারে না)
- Gradient flow এর জন্য tanh RNN এ বেশি উপযুক্ত
> তুলনামূলকভাবে — ANN এ ReLU বেশি ব্যবহার হয় কারণ সেখানে hidden state এর concept নেই।

### গুরুত্বপূর্ণ বিষয়:

> **Weights shared হয়।** Wₓ, Wₕ, Wᵧ — এই weight matrix গুলো পুরো sequence জুড়ে **একই** থাকে। প্রতিটা time step এ নতুন weight তৈরি হয় না। এটাই RNN কে efficient করে তোলে।

---

## ৮. RNN এর Hidden State — Memory কীভাবে কাজ করে

Hidden State হলো RNN এর **"মস্তিষ্ক"** — এটা সব আগের information এর একটা compressed summary।

### উদাহরণ:

ধরো sentence: `"The weather is very"`

RNN এটা process করে এভাবে:

```
h₀ = [0, 0, ..., 0]   ← শুরু, কিছু জানে না

Step 1: x₁ = "The"     → h₁ = tanh(Wₓ·x₁ + Wₕ·h₀ + b)
Step 2: x₂ = "weather" → h₂ = tanh(Wₓ·x₂ + Wₕ·h₁ + b)
Step 3: x₃ = "is"      → h₃ = tanh(Wₓ·x₃ + Wₕ·h₂ + b)
Step 4: x₄ = "very"    → h₄ = tanh(Wₓ·x₄ + Wₕ·h₃ + b)
```

Step 4 এ hidden state h₄ এর মধ্যে "The weather is very" — পুরো context টাই encode হয়ে আছে। তাই পরের word predict করতে পারে যে "cold" বা "hot" আসতে পারে।

> **h₀ (initial hidden state)** সাধারণত সব **zero** দিয়ে শুরু হয়।

**Hidden State এর size কে বলা হয় `hidden_size` বা `hidden_units`।** এটা একটা hyperparameter — তুমি নিজে ঠিক করো। বড় হলে বেশি তথ্য ধরতে পারে, কিন্তু computation বেশি।

---

## ৯. Unfolded RNN — Through Time বোঝো

RNN কে diagram এ সাধারণত **folded** দেখানো হয় — একটা loop এর মতো। কিন্তু এটা বোঝার সহজ উপায় হলো **unfold** করে দেখা।

```
Folded RNN:
        ┌─────┐
xₜ ──▶ │     │──▶ yₜ
        │ RNN │
   ┌──▶ │     │──┐
   │    └─────┘  │ (hidden state loop back করে)
   └─────────────┘


Unfolded RNN (through time):

         x₁           x₂           x₃           x₄
          │            │            │            │
          ▼            ▼            ▼            ▼
h₀ ──▶ [RNN] ──h₁──▶ [RNN] ──h₂──▶ [RNN] ──h₃──▶ [RNN] ──h₄──▶ output
         │            │            │            │
         ▼            ▼            ▼            ▼
        y₁           y₂           y₃           y₄
```

Unfolded করলে দেখা যায়:
- প্রতিটা step এ **একই RNN cell** repeat হচ্ছে (same weights)
- Hidden state বাম থেকে ডানে flow করছে — এটাই memory
- প্রতিটা step আগের step এর উপর নির্ভরশীল
**Gradient কীভাবে যায় (training এ)?**
Training এর সময় gradient ডান থেকে বামে যায়:
```
Loss ← h₄ ← h₃ ← h₂ ← h₁ ← h₀
```
এই process কে বলা হয় **BPTT (Backpropagation Through Time)**।

---